# Day 5 — API End-to-End Tests
Live tests against the running Defect Prediction Engine API.

In [ ]:
# Cell 1 — Imports and config
import json
import time
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

BASE_URL = "http://localhost:8000"
TIMEOUT = 300  # 5 minutes for full pipeline

print(f"Targeting API at: {BASE_URL}")
print("All systems ready — run cells in order")

In [ ]:
# Cell 2 — Health check
resp = requests.get(f"{BASE_URL}/health", timeout=10)
resp.raise_for_status()
health = resp.json()

print("=" * 50)
print("HEALTH CHECK")
print("=" * 50)
pprint(health)

assert health["status"] in ("healthy", "degraded"), (
    f"API is unhealthy: {health['status']}. "
    f"Models loaded: {health['models_loaded']}"
)

print(f"\n✓ Status     : {health['status']}")
print(f"✓ Version    : {health['version']}")
print(f"✓ Uptime     : {health['uptime_seconds']:.1f}s")
print(f"✓ MLflow     : {'connected' if health['mlflow_connected'] else 'not connected'}")
print(f"✓ Models     : {health['models_loaded']}")

In [ ]:
# Cell 3 — Analyze flask repo
print("Submitting analysis for pallets/flask ...")
print("(This may take 1-3 minutes for first run — repo needs to be cloned)")

t_start = time.time()
resp = requests.post(
    f"{BASE_URL}/analyze",
    json={
        "repo_url": "https://github.com/pallets/flask",
        "since_days": 365,
        "top_k": 20,
        "use_hybrid": True,
    },
    timeout=TIMEOUT,
)
elapsed = time.time() - t_start

if resp.status_code != 200:
    print(f"ERROR {resp.status_code}: {resp.text}")
    raise RuntimeError(f"Analyze failed: {resp.status_code}")

result = resp.json()
job_id = result["job_id"]

print(f"\n{'='*50}")
print(f"JOB ID        : {job_id}")
print(f"Status        : {result['status']}")
print(f"Repo          : {result['repo_name']}")
print(f"Model used    : {result['model_used']}")
print(f"Total files   : {result['total_files_analyzed']}")
print(f"Buggy pred.   : {result['buggy_files_predicted']}")
print(f"Model AUC     : {result['model_auc']}")
print(f"Precision@K   : {result['precision_at_k']}")
print(f"Elapsed (wall): {elapsed:.1f}s")
print()
print("Top 5 risky files:")
for r in result["top_k_results"][:5]:
    top_feat = r["top_shap_features"][0]["feature_name"] if r["top_shap_features"] else "N/A"
    print(f"  [{r['rank']:2d}] {r['risk_label']:6s}  {r['risk_score']:.3f}  {r['file_path']}  (top SHAP: {top_feat})")

if result.get("warnings"):
    print(f"\nWarnings: {result['warnings']}")

In [ ]:
# Cell 4 — Styled DataFrame of results
rows = []
for r in result["top_k_results"]:
    top_shap = r["top_shap_features"][0] if r["top_shap_features"] else None
    rows.append({
        "rank":         r["rank"],
        "file_path":    r["file_path"].split("/")[-1],  # basename for readability
        "full_path":    r["file_path"],
        "risk_score":   r["risk_score"],
        "risk_label":   r["risk_label"],
        "top_feature":  top_shap["feature_name"] if top_shap else "",
        "shap_value":   top_shap["shap_value"] if top_shap else 0.0,
        "loc":          r.get("lines_of_code") or 0,
        "days_ago":     r.get("last_modified_days_ago") or 0,
    })

df = pd.DataFrame(rows)

color_map = {"HIGH": "#ff4444", "MEDIUM": "#ff9900", "LOW": "#22bb33"}

def highlight_risk(val):
    color = color_map.get(val, "black")
    return f"color: {color}; font-weight: bold"

styled = (
    df[["rank", "file_path", "risk_score", "risk_label", "top_feature", "shap_value", "loc", "days_ago"]]
    .style
    .applymap(highlight_risk, subset=["risk_label"])
    .background_gradient(subset=["risk_score"], cmap="RdYlGn_r", vmin=0, vmax=1)
    .set_caption(f"Top {len(df)} Risky Files — {result['repo_name']}")
    .format({"risk_score": "{:.3f}", "shap_value": "{:+.4f}"})
)
styled

In [ ]:
# Cell 5 — Explain the riskiest file
top_file = result["top_k_results"][0]["file_path"]
print(f"Explaining: {top_file}")

resp_ex = requests.get(
    f"{BASE_URL}/explain/{job_id}/{top_file}",
    timeout=30,
)
resp_ex.raise_for_status()
explain = resp_ex.json()

print(f"\n{'='*60}")
print("PLAIN ENGLISH SUMMARY")
print("=" * 60)
print(explain["plain_english_summary"])

print(f"\nSimilar files (by risk score):")
for sf in explain["similar_files"]:
    print(f"  {sf}")

print(f"\nGNN embedding neighbors:")
for nb in explain.get("embedding_neighbors", ["(none — GNN not loaded)"]):
    print(f"  {nb}")

# Plot SHAP waterfall
waterfall = explain["shap_waterfall"][:15]  # top 15 for readability
if waterfall:
    feats  = [w["feature_name"] for w in waterfall]
    values = [w["shap_value"]   for w in waterfall]
    colors = ["#d73027" if v > 0 else "#4575b4" for v in values]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(feats[::-1], values[::-1], color=colors[::-1], edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("SHAP value (impact on defect risk)", fontsize=11)
    ax.set_title(
        f"SHAP Waterfall — {top_file.split('/')[-1]}\n"
        f"Risk score: {explain['risk_score']:.3f} ({explain['risk_label']})",
        fontsize=12, fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig("reports/figures/day5_shap_waterfall.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nSHAP waterfall saved to reports/figures/day5_shap_waterfall.png")

In [ ]:
# Cell 6 — Timing breakdown bar chart
timing = {
    "Mining": result["mining_time_ms"],
    "Feature\nextraction": result["feature_time_ms"],
    "Prediction": result["prediction_time_ms"],
}
total = result["analysis_time_ms"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors_timing = ["#2196F3", "#4CAF50", "#FF9800"]
bars = ax1.bar(
    list(timing.keys()),
    list(timing.values()),
    color=colors_timing,
    edgecolor="white",
    linewidth=1.5,
)
for bar, val in zip(bars, timing.values()):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f"{val:,.0f}ms",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )
ax1.set_ylabel("Time (ms)", fontsize=11)
ax1.set_title("Pipeline Timing Breakdown", fontsize=12, fontweight="bold")
ax1.set_ylim(0, max(timing.values()) * 1.2)

# Pie chart
ax2.pie(
    list(timing.values()),
    labels=list(timing.keys()),
    colors=colors_timing,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.7,
)
ax2.set_title(f"Time Distribution\n(total: {total:,.0f}ms)", fontsize=12, fontweight="bold")

plt.suptitle(
    f"Day 5 — Pipeline Timing | Repo: {result['repo_name']} | Files: {result['total_files_analyzed']}",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.savefig("reports/figures/day5_timing.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Timing chart saved to reports/figures/day5_timing.png")
print(f"\nTotal analysis time: {total:,.0f}ms ({total/1000:.1f}s)")

In [ ]:
# Cell 7 — MLflow experiments
resp_exp = requests.get(f"{BASE_URL}/experiments", timeout=15)

if resp_exp.status_code == 200:
    exp_data = resp_exp.json()
    print(f"Experiment: {exp_data['experiment_name']}")
    print(f"Total runs: {exp_data['total_runs']}")
    print(f"Best run  : {exp_data['best_run']['run_name']} (AUC={exp_data['best_run']['auc']:.4f})")

    runs_df = pd.DataFrame(exp_data["all_runs"])
    runs_df = runs_df.sort_values("auc", ascending=False).reset_index(drop=True)

    styled_runs = (
        runs_df[["run_name", "auc", "f1", "precision_at_20", "n_features", "timestamp"]]
        .style
        .background_gradient(subset=["auc", "f1", "precision_at_20"], cmap="YlGn")
        .set_caption("MLflow Experiment Runs")
        .format({"auc": "{:.4f}", "f1": "{:.4f}", "precision_at_20": "{:.4f}"})
    )
    display(styled_runs)
elif resp_exp.status_code == 404:
    print("No MLflow runs found yet — run Days 3+4 scripts to populate.")
    print(f"Detail: {resp_exp.json().get('detail', '')}")
else:
    print(f"Experiments endpoint returned {resp_exp.status_code}: {resp_exp.text}")

In [ ]:
# Cell 8 — End-to-end summary
print("\n" + "="*65)
print("DAY 5 END-TO-END TEST SUMMARY")
print("="*65)

tests = [
    ("GET /health",                  health.get("status") in ("healthy", "degraded")),
    ("POST /analyze (flask)",         result.get("status") == "completed"),
    ("GET /analyze/{job_id}",         job_id is not None),
    ("GET /explain/{job_id}/{file}",  explain.get("plain_english_summary") is not None and len(explain["plain_english_summary"]) > 0),
    ("GET /experiments",              resp_exp.status_code in (200, 404)),  # 404 ok if no runs
]

all_passed = True
for name, passed in tests:
    status_str = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {status_str}  {name}")
    if not passed:
        all_passed = False

print()
print(f"Pipeline timing:")
print(f"  Mining       : {result['mining_time_ms']:>8,.0f} ms")
print(f"  Features     : {result['feature_time_ms']:>8,.0f} ms")
print(f"  Prediction   : {result['prediction_time_ms']:>8,.0f} ms")
print(f"  Total        : {result['analysis_time_ms']:>8,.0f} ms")
print()
print(f"Model performance:")
print(f"  AUC          : {result['model_auc']}")
print(f"  Precision@K  : {result['precision_at_k']}")
print(f"  Files analyzed: {result['total_files_analyzed']}")
print()

if all_passed:
    print("ALL TESTS PASSED ✓")
    print()
    print("Day 5 complete — API is production-ready.")
    print("Next: git commit and push to wrap up the project.")
else:
    print("SOME TESTS FAILED — check output above")
print("="*65)